# 04 model export and verification

**Owner:** Denis. Original notebook cells: 44, 45, 46, 47.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
import tempfile
import shutil
import zipfile

from sklearn.metrics import f1_score

# Save the selected MLP and its configuration
model_dir = (
    "/Volumes/workspace/default/"
    "eye_style_team2/models/mlp_v1"
)

os.makedirs(model_dir, exist_ok=True)

# Retrieve the selected model's actual validation results
selected_model_row = tuning_df.loc[
    tuning_df["Model"] == best_model_name
].iloc[0]

expected_f1 = float(selected_model_row["F1"])
expected_threshold = float(selected_model_row["Threshold"])

# Confirm the selected classification threshold
assert np.isclose(
    final_mlp_threshold,
    expected_threshold
), "Selected classification threshold does not match."

# Verify the selected model before saving
selected_probs = best_mlp_model.predict(
    X_val_nn,
    batch_size=256,
    verbose=0
).ravel()

selected_preds = (
    selected_probs >= final_mlp_threshold
).astype(int)

verified_f1 = f1_score(
    y_val_nn,
    selected_preds,
    zero_division=0
)

print("Selected model:", best_model_name)
print("Selected threshold:", final_mlp_threshold)
print("Expected validation F1:", round(expected_f1, 4))
print("Verified validation F1:", round(verified_f1, 4))
print("F1 difference:", round(
    abs(verified_f1 - expected_f1), 6
))

# Validate against current optimization results
assert abs(verified_f1 - expected_f1) < 0.002, (
    "Validation F1 differs from the selected model's "
    "recorded optimization result. Check the model "
    "before saving."
)

print("Model validation check passed.")

# Recover the actual selected model architecture
architecture = [
    int(layer.units)
    for layer in best_mlp_model.layers
    if isinstance(layer, tf.keras.layers.Dense)
]

dropout_rates = [
    float(layer.rate)
    for layer in best_mlp_model.layers
    if isinstance(layer, tf.keras.layers.Dropout)
]

learning_rate = float(
    tf.keras.backend.get_value(
        best_mlp_model.optimizer.learning_rate
    )
)

# Define final model and metadata paths
model_path = f"{model_dir}/no_show_mlp_v1.keras"
metadata_path = f"{model_dir}/mlp_metadata.json"

# Save model settings and deployment information
model_info = {
    "model_version": "mlp_v1",
    "model_name": best_model_name,
    "architecture": architecture,
    "dropout": (
        dropout_rates[0] if dropout_rates else 0.0
    ),
    "dropout_rates": dropout_rates,
    "learning_rate": learning_rate,
    "batch_size": 64,
    "max_epochs": 100,
    "early_stopping_patience": 10,
    "input_dimensions": int(X_train_nn.shape[1]),
    "threshold": float(final_mlp_threshold),
    "validation_f1": float(verified_f1),
    "expected_validation_f1": expected_f1,
    "feature_columns": list(features),
    "positive_class": "no_show",
    "prediction_time": "booking",
    "preprocessor_path": (
        "/Volumes/workspace/default/"
        "eye_style_team2/models/baseline/"
        "logistic_regression.joblib"
    ),
    "tensorflow_version": tf.__version__
}

# Save locally first to avoid Unity Catalog
# unsupported random-write operations
with tempfile.TemporaryDirectory() as temp_dir:

    local_model_path = os.path.join(
        temp_dir, "no_show_mlp_v1.keras"
    )

    local_metadata_path = os.path.join(
        temp_dir, "mlp_metadata.json"
    )

    # Save Keras model to local temporary storage
    best_mlp_model.save(local_model_path)

    # Verify local model file
    assert os.path.isfile(local_model_path)
    assert os.path.getsize(local_model_path) > 0
    assert zipfile.is_zipfile(local_model_path)

    print("Local model export successful.")

    # Save metadata locally
    with open(local_metadata_path, "w") as f:
        json.dump(model_info, f, indent=2)

    # Copy model and metadata to Unity Catalog
    shutil.copyfile(local_model_path, model_path)
    shutil.copyfile(local_metadata_path, metadata_path)

    # Verify copied model
    assert os.path.isfile(model_path)

    assert (
        os.path.getsize(model_path)
        == os.path.getsize(local_model_path)
    ), "Model copy verification failed."

    # Verify copied metadata
    assert os.path.isfile(metadata_path)

    assert (
        os.path.getsize(metadata_path)
        == os.path.getsize(local_metadata_path)
    ), "Metadata copy verification failed."

    print("Unity Catalog copy verification passed.")

# Verify saved metadata contents
with open(metadata_path, "r") as f:
    saved_metadata = json.load(f)

assert saved_metadata == model_info, (
    "Saved metadata verification failed."
)

# Final export summary
print("\nModel saved:", model_path)
print("Metadata saved:", metadata_path)

print(
    "Model file size:",
    round(os.path.getsize(model_path) / 1024, 2),
    "KB"
)

print("\n=== SELECTED MODEL CONFIGURATION ===")
print(json.dumps(model_info, indent=2))

print("\nModel export completed successfully.")

In [ ]:

import os

volume_dir = "/Volumes/workspace/default/eye_style_team2/models/mlp_v1"

os.makedirs(volume_dir, exist_ok=True)

test_path = f"{volume_dir}/write_test.txt"

with open(test_path, "w") as f:
    f.write("Databricks Volume write test successful")

with open(test_path, "r") as f:
    print(f.read())

os.remove(test_path)

print("Volume read/write test passed.")


In [ ]:
import mlflow
import mlflow.tensorflow

mlflow.set_experiment("/Shared/Eye_Style_No_Show_Models")

with mlflow.start_run(run_name="eye_style_mlp_v1") as run:
    mlflow.log_params({
        "architecture": str(model_info["architecture"]),
        "dropout": model_info["dropout"],
        "learning_rate": model_info["learning_rate"],
        "batch_size": 64,
        "reference_mlp_epochs_completed": len(history.history["loss"]),
        "classification_threshold": float(final_mlp_threshold)
    })

    mlflow.log_metrics({
        "validation_f1": float(selected_model_row["F1"]),
        "validation_roc_auc": float(selected_model_row["ROC-AUC"]),
        "validation_pr_auc": float(selected_model_row["PR-AUC"])
    })

    mlflow.tensorflow.log_model(
        best_mlp_model,
        name="no_show_mlp"
    )

    print("MLflow run ID:", run.info.run_id)
    print("Model URI:", f"runs:/{run.info.run_id}/no_show_mlp")

In [ ]:
import mlflow
import numpy as np
from sklearn.metrics import f1_score

# Retrieve the saved run and model
run_id = run.info.run_id
model_uri = f"runs:/{run_id}/no_show_mlp"

print("Model URI:", model_uri)

# Load model from MLflow
loaded_mlp = mlflow.tensorflow.load_model(model_uri)

# Verify predictions match the original model
original_probs = best_mlp_model.predict(
    X_val_nn, batch_size=256, verbose=0
).ravel()

loaded_probs = loaded_mlp.predict(
    X_val_nn, batch_size=256, verbose=0
).ravel()

max_difference = np.max(
    np.abs(original_probs - loaded_probs)
)

loaded_predictions = (
    loaded_probs >= final_mlp_threshold
).astype(int)

loaded_f1 = f1_score(
    y_val_nn,
    loaded_predictions,
    zero_division=0
)

print("Maximum probability difference:", max_difference)
print("Loaded model validation F1:", round(loaded_f1, 4))

assert np.allclose(original_probs, loaded_probs, atol=1e-6)
print("Saved model verified successfully.")
# Persist the verified run and its matching model metadata for the inference stage.
with open(f"{model_dir}/verified_run.json", "w", encoding="utf-8") as f:
    json.dump({"model_uri": model_uri, "threshold": float(final_mlp_threshold), "model_name": best_model_name, "model_version": "eye_style_mlp_v1", "preprocessor_path": model_info["preprocessor_path"]}, f, indent=2)
